# Thematic Analysis of Repository Artifacts

This notebook summarizes the thematic analysis of **repository artifacts**
associated with the selected modernization commits.

The qualitative analysis was performed using **ATLAS.ti**. The analyzed
artifacts include repository evidence such as commit messages and pull
request (PR) discussions. ATLAS.ti was used to identify and organize
relevant quotes into codes, broader code groups, and themes.

This notebook does **not** perform the qualitative coding itself. Instead,
it processes the ATLAS.ti export and generates descriptive summaries and
visualizations of the resulting coding structure.

The analysis focuses on three levels:

- **Codes** — specific concepts identified in the repository artifacts.
- **Groups** — broader categories used to organize related codes.
- **Themes** — higher-level concepts emerging from the coded evidence.

The resulting tables and figures support the qualitative results reported
in the paper.


## 1. Setup


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Display configuration
# ---------------------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


# ---------------------------------------------------------------------
# Input and output paths
# ---------------------------------------------------------------------
# Update INPUT_FILE if the ATLAS.ti export is stored elsewhere.
#
# Generated results are kept in a separate directory so that source data
# are never overwritten.

INPUT_FILE = Path(
    "/content/drive/MyDrive/Documents/unb/TA-commits-PRs2.csv"
)

OUTPUT_DIR = Path("/content/repository_artifacts_ta_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Publication-oriented Matplotlib configuration
# ---------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR}")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file was not found: {INPUT_FILE}\n"
        "Update INPUT_FILE in the setup cell."
    )

# The original ATLAS.ti export uses semicolon-separated fields and
# latin-1 encoding.
df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="latin-1",
)

print(f"Loaded {len(df):,} coded excerpts.")
print(f"Columns: {list(df.columns)}")


## 2. Data Validation


In [ ]:
# Columns required by the analysis below.
REQUIRED_COLUMNS = {"themes", "groups", "codes"}

missing_columns = REQUIRED_COLUMNS - set(df.columns)

if missing_columns:
    raise ValueError(
        "The ATLAS.ti export is missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("Dataset dimensions")
print(f"  Rows:    {len(df):,}")
print(f"  Columns: {len(df.columns):,}")

print("\nMissing values:")
display(
    df[list(REQUIRED_COLUMNS)]
    .isna()
    .sum()
    .rename("missing")
    .to_frame()
)

print("\nUnique values:")
for column in ["groups", "themes", "codes"]:
    print(f"  {column}: {df[column].nunique(dropna=True):,}")


## 3. Normalize Labels

The labels below are presentation-oriented names used consistently in the
analysis and figures.

The original ATLAS.ti labels are retained in the source CSV. A copy of the
data is used here so that the original export is not modified.


In [ ]:
# ---------------------------------------------------------------------
# Group labels
# ---------------------------------------------------------------------

GROUP_RENAMES = {
    "Tooling & Codebase Infrastructure":
        "Modernization Infrastructure",
    "Type System & Static Analysis":
        "Type System Evolution",
    "Code Structure & Design":
        "Codebase Design Evolution",
    "Software Correctness & Reliability":
        "Software Reliability",
    "Code Quality Maintenance":
        "Code Quality",
}


# ---------------------------------------------------------------------
# Theme labels
# ---------------------------------------------------------------------

THEME_RENAMES = {
    "Transformation Strategies": "Practices",
}


# Work on a copy of the ATLAS.ti export.
df_analysis = df.copy()

df_analysis["groups"] = (
    df_analysis["groups"]
    .replace(GROUP_RENAMES)
)

df_analysis["themes"] = (
    df_analysis["themes"]
    .replace(THEME_RENAMES)
)

print("Group labels after normalization:")
display(
    df_analysis["groups"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .to_frame("group")
)

print("Theme labels after normalization:")
display(
    df_analysis["themes"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .to_frame("theme")
)


## 4. Coding Structure


In [ ]:
# Display the unique combinations of group, theme, and code.
#
# This provides a compact view of the hierarchical coding structure
# exported from ATLAS.ti.

coding_structure = (
    df_analysis[["groups", "themes", "codes"]]
    .drop_duplicates()
    .sort_values(["groups", "themes", "codes"])
    .reset_index(drop=True)
)

display(coding_structure)


## 5. Code Frequencies


In [ ]:
# Each row corresponds to a coded quote/excerpt in the ATLAS.ti export.
# Therefore, counting rows provides the frequency of each code.
#
# This is a frequency of coded excerpts, not necessarily a frequency of
# unique commits, PRs, developers, or artifacts.

code_counts = (
    df_analysis
    .groupby(["groups", "themes", "codes"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(code_counts)


### 5.1 Top Codes


In [ ]:
top_codes = code_counts.head(10).copy()

display(top_codes)


In [ ]:
plot_data = top_codes.sort_values("count", ascending=True)

fig, ax = plt.subplots(
    figsize=(7.0, 4.2),
    constrained_layout=True,
)

ax.barh(
    plot_data["codes"],
    plot_data["count"],
)

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=3,
        fontsize=8,
    )

plt.show()


## 6. Group Frequencies


In [ ]:
# Count coded excerpts associated with each group.

group_counts = (
    df_analysis
    .groupby("groups", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(group_counts)


In [ ]:
plot_data = group_counts.sort_values("count", ascending=True)

fig, ax = plt.subplots(
    figsize=(7.0, 4.0),
    constrained_layout=True,
)

ax.barh(
    plot_data["groups"],
    plot_data["count"],
)

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=3,
        fontsize=8,
    )

plt.show()


## 7. Theme Frequencies


In [ ]:
# Count coded excerpts associated with each theme.

theme_counts = (
    df_analysis
    .groupby("themes", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(theme_counts)


In [ ]:
plot_data = theme_counts.sort_values("count", ascending=True)

fig, ax = plt.subplots(
    figsize=(7.0, max(4.0, 0.42 * len(plot_data))),
    constrained_layout=True,
)

ax.barh(
    plot_data["themes"],
    plot_data["count"],
)

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=3,
        fontsize=8,
    )

plt.show()


## 8. Group–Theme Distribution


In [ ]:
# Count coded excerpts for every group/theme combination.
group_theme_counts = (
    df_analysis
    .groupby(["groups", "themes"], dropna=False)
    .size()
    .reset_index(name="count")
)

display(
    group_theme_counts.sort_values(
        ["groups", "count"],
        ascending=[True, False],
    )
)


### 8.1 Theme Frequency Within Each Group


In [ ]:
# Rank themes within each group.
group_theme_ranking = (
    group_theme_counts
    .sort_values(
        ["groups", "count"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(group_theme_ranking)


## 9. Group–Theme Heatmap


The heatmap shows how coded excerpts are distributed across groups and
themes.

Rows represent the broader groups, columns represent themes, and each cell
contains the number of coded excerpts associated with that combination.

The visualization uses Matplotlib directly so that the notebook has no
additional visualization dependency such as Seaborn.


In [ ]:
# Build the group-by-theme contingency table.
pivot = (
    df_analysis
    .groupby(["groups", "themes"])
    .size()
    .unstack(fill_value=0)
)

# Order groups by their total number of coded excerpts.
pivot = pivot.loc[
    pivot.sum(axis=1)
    .sort_values(ascending=False)
    .index
]

# Order themes by their overall frequency.
pivot = pivot[
    pivot.sum(axis=0)
    .sort_values(ascending=False)
    .index
]

display(pivot)


In [ ]:
fig, ax = plt.subplots(
    figsize=(7.0, 5.5),
    constrained_layout=True,
)

# imshow provides the heatmap without requiring Seaborn.
image = ax.imshow(
    pivot.values,
    aspect="auto",
)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(
    pivot.columns,
    rotation=45,
    ha="right",
)

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

ax.set_xlabel("")
ax.set_ylabel("Group")

# Annotate every cell with its frequency.
for row in range(pivot.shape[0]):
    for column in range(pivot.shape[1]):
        value = pivot.iloc[row, column]
        ax.text(
            column,
            row,
            f"{value:d}",
            ha="center",
            va="center",
            fontsize=8,
        )

# Add a color scale without prescribing a particular palette.
colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label("Frequency")

figure_path = OUTPUT_DIR / "repository_artifacts_ta.pdf"

fig.savefig(
    figure_path,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved to: {figure_path}")


## 10. Theme–Code Mapping


In [ ]:
# List the codes associated with each group/theme combination.

theme_code_mapping = (
    df_analysis
    .groupby(["groups", "themes"])["codes"]
    .agg(lambda values: sorted(set(values.dropna())))
    .reset_index()
)

theme_code_mapping["code_count"] = (
    theme_code_mapping["codes"].apply(len)
)

display(theme_code_mapping)


## 11. Export Aggregated Results


The following files contain aggregated results derived from the ATLAS.ti
export. They do not modify the source qualitative data.


In [ ]:
outputs = {
    "code_frequencies.csv": code_counts,
    "group_frequencies.csv": group_counts,
    "theme_frequencies.csv": theme_counts,
    "group_theme_frequencies.csv": group_theme_counts,
    "theme_code_mapping.csv": theme_code_mapping,
    "group_theme_matrix.csv": pivot.reset_index(),
}

for filename, data in outputs.items():
    path = OUTPUT_DIR / filename
    data.to_csv(path, index=False)

print("Generated files:")
for filename in outputs:
    print(f"  - {OUTPUT_DIR / filename}")

print(f"  - {OUTPUT_DIR / 'repository_artifacts_ta.pdf'}")


## 12. Summary


In [ ]:
print("Repository-artifact thematic analysis")
print("=" * 45)

print(f"Total coded excerpts: {len(df_analysis):,}")
print(f"Unique codes:        {df_analysis['codes'].nunique():,}")
print(f"Unique groups:       {df_analysis['groups'].nunique():,}")
print(f"Unique themes:       {df_analysis['themes'].nunique():,}")

print("\nFrequency by group:")
display(group_counts)

print("\nFrequency by theme:")
display(theme_counts)


## 13. Reproducibility Notes

- The qualitative coding was conducted in **ATLAS.ti**.
- The input to this notebook is the ATLAS.ti export containing coded quotes
  from repository artifacts.
- This notebook performs **aggregation, inspection, and visualization** of
  the coding results; it does not reproduce the manual coding decisions.
- Frequencies represent coded excerpts in the export. They should not be
  interpreted as counts of unique developers, commits, or PRs unless the
  source dataset has that one-to-one structure.
- The original ATLAS.ti export should be handled according to the study's
  privacy and data-sharing requirements.
- Generated CSV files and figures can be included in the replication
  package when appropriate.
